# Query a Foundry IQ knowledge base through MCP

Connect directly to the Azure AI Search MCP endpoint for the minimal-reasoning `contoso-company-kb-minimal` knowledge base, inspect the tool it exposes, and retrieve grounded HR and benefits content. The endpoint uses the signed-in Azure Developer CLI identity and requires Search data-plane access.

## Step 1: Load configuration and create an MCP session helper

Load the Azure tenant, Search endpoint, and minimal knowledge-base name from the project `.env` file. The MCP URL targets the knowledge base directly and uses an Azure AI Search bearer token during session initialization.

In [1]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
SEARCH_ENDPOINT = os.environ["AZURE_AI_SEARCH_SERVICE_ENDPOINT"].rstrip("/")
KNOWLEDGE_BASE_NAME = os.environ.get(
    "AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME",
    "contoso-company-kb-minimal",
)
SEARCH_SCOPE = "https://search.azure.com/.default"
KNOWLEDGE_BASE_MCP_URL = (
    f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}"
    "/mcp?api-version=2026-05-01-preview"
)


@asynccontextmanager
async def open_knowledge_base_session():
    credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
    token = credential.get_token(SEARCH_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(
            KNOWLEDGE_BASE_MCP_URL,
            http_client=http_client,
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print(f"Foundry IQ MCP configured for {KNOWLEDGE_BASE_NAME}")

Foundry IQ MCP configured for contoso-company-kb-minimal


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the minimal Foundry IQ knowledge base, including its detailed retrieval guidance and input schema.

In [2]:
from rich.console import Console

console = Console()

async with open_knowledge_base_session() as session:
    tools_response = await session.list_tools()

knowledge_base_tools = tools_response.tools

for tool in knowledge_base_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

Tool: knowledge_base_retrieve

Description: Use knowledge_base_retrieve to search for authoritative, source-attributable information or documents.
This knowledge base is always relevant to the user and any organizations they're affiliated with. You may call this
tool with ambiguous queries to retrieve relevant context before asking the user for further clarification. If a 
Knowledge Base Description is provided, use it to understand what content is included/excluded and to scope your 
retrieval.

Populate queries by converting user inputs into concise, self-contained search queries anchored by named entities, 
IDs, and specific terms. Use the user's own words when rewriting, preserving entities, references, identifiers, 
ranges, aliases, filters, exact terms, grouping directives, and comparison directives. For temporal, as-of, date 
range, fiscal year, quarter, and recency constraints, preserve the exact time constraint in the query. Query in the
user's language.

If prior retrieval is insufficient, use shorter keyword queries, separate searches for key entities, or expanded 
constraints.
Knowledge Base Description - Contains internal HR documents about employee benefits and health/wellness programs.

Input schema:

{
  "type": "object",
  "properties": {
    "queries": {
      "description": "1 to 3 search queries. Each query: max 150 characters; under 12 words. Cover distinct retrieval goals and alternate evidence wording. Do not add duplicates that only reorder terms.\n\nAlways return a JSON array of strings, even when issuing a single query. Example: {\"queries\": [\"contoso fiscal year 2024 revenue\"]}.",
      "type": "array",
      "items": {
        "type": "string",
        "maxLength": 150
      },
      "minItems": 1,
      "maxItems": 3
    }
  },
  "required": [
    "queries"
  ],
  "additionalProperties": false
}

## Step 3: Retrieve from the minimal knowledge base

Call `knowledge_base_retrieve` with one to three concise search queries. The minimal-reasoning knowledge base returns extractive, source-attributable results rather than synthesizing a final answer.

In [4]:
import json

queries = [
    "employee health plan benefits enrollment",
    "benefits enrollment deadlines",
]

async with open_knowledge_base_session() as session:
    result = await session.call_tool(
        "knowledge_base_retrieve",
        {"queries": queries},
    )

responses = [block.text for block in result.content if block.type == "text"]
for response in responses:
    console.print_json(data=json.loads(response))

[
  {
    "ref_id": 0,
    "content": "you minimum essential coverage or face a penalty. \n\nFollowing the law is an important part of employee benefits, and Contoso and Northwind \n\nHealth strive to ensure that the Northwind Standard plan is in compliance with all \n\napplicable laws. Employees should make sure they understand their rights and \n\nresponsibilities under the law when it comes to their employer-provided health insurance \n\nplan. With Northwind Standard, you can be sure that you’re getting the coverage you need \n\nand that you’re in compliance with the law. \n\nEntire Contract \n\nOTHER INFORMATION ABOUT THIS PLAN: Entire Contract \n\nThe Northwind Standard plan is a contract between the employee and Northwind Health. \n\nBy enrolling in the plan, the employee agrees to all of the terms and conditions included in \n\nthe plan documents. It is important to understand that the plan documents are the ultimate \n\nauthority for any questions about benefits, coverage, and exclusions. \n\nThe plan documents state that the Northwind Standard plan provides coverage for medical, \n\nvision, and dental services. This coverage includes preventive care services and prescription \n\ndrug coverage. The plan does not provide coverage for emergency services, mental health \n\nand substance abuse coverage, or out-of-network services. \n\nThe plan documents also include information on how to access care, including a list of in- \n\nnetwork providers such as primary care physicians, specialists, hospitals, and pharmacies. \n\nAdditionally, the plan documents outline the plan’s coordination of benefits and any \n\nlimitations or exclusions. \n\nIt is important to remember that the plan documents are the ultimate authority for any \n\nquestions about benefits, coverage, and exclusions. If there is ever a discrepancy between \n\nwhat is stated in the plan documents and what is stated in any other sources, such as \n\nNorthwind Health’s website or a customer service representative, the plan documents take \n\nprecedence. \n\nTips for Employees"
  },
  {
    "ref_id": 1,
    "content": "that may apply. Here are some tips that can help: \n\n• Make sure you understand what type of coverage you have. This will help you understand \n\nwhich provider is the primary payer and which is the secondary payer. \n\n• Keep track of all your medical expenses and bills. This will help you understand how much \n\nyou need to pay out of pocket, and how much the primary and secondary payers will cover. \n\n• Make sure you understand the rules and regulations of each coverage plan. This will help \n\nyou understand when claims will be covered and what benefits you are eligible for. \n\n• Know the deadlines for filing claims. This will help you ensure that you get the coverage \n\nyou need and that your claims are processed in a timely manner. \n\n• Ask questions if you are unsure about anything related to the Northwind Health Plus plan. \n\nThe customer service representatives at Northwind Health can help you understand the \n\nprimary and secondary rules, as well as any exceptions that may apply. \n\nIt’s important to understand the primary and secondary rules of the Northwind Health Plus \n\nplan, and to understand any exceptions that may apply. Following these tips can help you \n\nget the coverage you need and ensure that your claims are processed in a timely manner. \n\nCOB's Effect On Benefits \n\nWHAT IF I HAVE OTHER COVERAGE? - COB's Effect On Benefits \n\nWhen you have more than one health insurance policy, the policies coordinate benefits \n\nthrough a process called Coordination of Benefits (COB). Coordination of Benefits is a \n\nprocess that helps to determine which plan pays first when there are multiple health plans \n\n  \n\n\n\navailable. This process is important because it affects how much you will pay out-of-pocket \n\nfor care. \n\nWhen Northwind Health Plus is the primary insurance (the plan that pays benefits first), \n\nany oth